## S3 Data Trigger for SageMaker Pipeline

This notebook demonstrates a simple event-driven MLOps workflow:

New CSV uploaded to S3
→ EventBridge detects the upload
→ Lambda starts SageMaker Pipeline
→ SageMaker Pipeline trains and evaluates a model
→ Quality gate checks the evaluation metric
The notebook is organised so it can be rerun from top to bottom. It creates a separate triggered pipeline:

iti113-team14-airbnb-instant-booking
This keeps it separate from the earlier manual pipeline:

iti113-team14-airbnb-instant-booking

## AWS Setup Required Before Running This Notebook
Before running this notebook, the AWS environment must already be prepared for the team. The team’s SageMaker Studio user/profile should use an execution role with permission to run SageMaker Processing Jobs, Training Jobs, Pipelines, and Model Registry operations. The role must also have S3 read/write access to the team’s assigned prefix, for example s3://nyp-26s1-iti113/iti113/team01/.

On the event-driven side, the S3 bucket must have EventBridge notifications enabled. An EventBridge rule should be configured to listen for new CSV files uploaded to the team trigger folder, for example iti113/team01/trigger/input/. The EventBridge rule should invoke a Lambda function.

The Lambda function needs permission to start the SageMaker Pipeline. It should read the uploaded S3 object path from the S3 event and pass that path into the pipeline as the InputDataUrl parameter.

CloudWatch Logs should also be available for troubleshooting. Lambda logs show whether the S3 event was received and whether the pipeline was started successfully. SageMaker Processing and Training logs show errors from preprocessing, training, and evaluation scripts.

This notebook assumes those AWS-side services have already been created and focuses on building, updating, testing, and verifying the S3-triggered SageMaker Pipeline.

## How to Use This Notebook
Run the sections in order:

Configure AWS and SageMaker settings
Write the pipeline scripts
Build and upsert the SageMaker Pipeline
Create and upload a synthetic Heart CSV file
Test the pipeline manually
Test the S3 → EventBridge → Lambda trigger
The final S3 trigger test assumes the EventBridge rule and Lambda function have already been created. The Lambda function should start:

iti113-team14-airbnb-instant-booking-triggered
and pass the uploaded S3 object URI as the InputDataUrl pipeline parameter.

## 1. Configure AWS and Project Settings
Edit only the values in this cell when reusing the notebook for another team or student.

Important classroom setting:

All SageMaker SDK uploads must stay under iti113/team01/...
This avoids S3 AccessDenied errors caused by writing outside the team prefix.

In [1]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import sagemaker
import json
import os
import time
from pathlib import Path


boto_session = boto3.Session()
region = boto_session.region_name or 'us-east-1'
s3 = boto_session.client('s3')
# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
# region  = boto3.Session().region_name

# Use the ITI113 course bucket and team prefix.
# This matches the team execution role S3 policy, e.g.:
# s3://nyp-26s1-iti113/iti113/team40/
BUCKET  = "nyp-26s1-iti113"

# Change these for the current student/profile.
TEAM_ID = "team14"
STUDENT_ID = "s1401"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "airbnb-instant-booking"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

MODEL_PACKAGE_GROUP_NAME = f"{TEAM_ID}-airbnb-instant-booking-Triggered"

# Local synthetic dataset filename
LOCAL_LISTING_FILE = "SyntheticListing.csv"

MANUAL_TEST_PREFIX = f"iti113/{TEAM_ID}/manual-input"
TRIGGER_PREFIX = f"iti113/{TEAM_ID}/trigger/input"

# Instance types.
# If ml.m5.large quota is 0, change these to an approved available training/processing type.
# For ITI113, keep Studio spaces on ml.t3.medium and use SageMaker jobs for training/processing.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"


PIPELINE_NAME       = f"iti113-{TEAM_ID}-airbnb-instant-booking-trigger"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-airbnb-instant-booking-Trigger"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-airbnb-instant-booking"
QUALITY_GATE_AUC    = 0.75

RAW_DATA_URI = f"s3://{BUCKET}/{PREFIX}/raw/Listings.csv"
SYNTHETIC_DATA_URI = f"s3://{BUCKET}/{PREFIX}/raw/SyntheticListing.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"



# Store the pipeline source files in S3 first, then download them into a clean local folder.
SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Pipeline                : iti113-team14-airbnb-instant-booking-trigger
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team14/data/airbnb-instant-booking
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team14
Pipeline source S3 URI  : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src
Local pipeline source   : pipeline_src


## 2. Define Pipeline Parameters

In [3]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

from sagemaker.workflow.parameters import (
    ParameterFloat,
    ParameterInteger,
    ParameterString,
)

pipeline_session = PipelineSession()
p_input_data = ParameterString(name="InputDataUrl", default_value=SYNTHETIC_DATA_URI)
# -------------------------------------------------------------------------
# Pipeline parameters — can be overridden at execution time
# -------------------------------------------------------------------------
p_n_est = ParameterInteger(name="NEstimators", default_value=150)
p_depth = ParameterInteger(name="MaxDepth", default_value=20)
p_features = ParameterString(name="MaxFeatures", default_value="sqrt")
p_min_leaf = ParameterInteger(name="MinSamplesLeaf", default_value=2)
p_min_split = ParameterInteger(name="MinSamplesSplit", default_value=5)
p_n_jobs = ParameterInteger(name="NJobs", default_value=2)
p_rand_state = ParameterInteger(name="RandomState", default_value=42)

# Evaluation metric threshold (optional)
p_gate = ParameterFloat(name="QualityGateAUC", default_value=QUALITY_GATE_AUC)

print("Pipeline parameters defined.")

Pipeline parameters defined.


## 4. Ensure the Model Package Group Exists

In [4]:
import botocore
session = sagemaker.Session()

region = boto3.Session().region_name

sm = boto3.client(
    "sagemaker",
    region_name=region,
)
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
        ModelPackageGroupDescription=(
            f"Model package group for {TEAM_ID} S3-triggered airbnb-instant-booking pipeline"
        )
    )
    print("Created model package group:", MODEL_PACKAGE_GROUP_NAME)

except botocore.exceptions.ClientError as e:
    error_code = e.response.get("Error", {}).get("Code", "")
    error_message = e.response.get("Error", {}).get("Message", "")

    if "already exists" in error_message.lower() or error_code == "ValidationException":
        print("Model package group already exists:", MODEL_PACKAGE_GROUP_NAME)
    else:
        raise

Model package group already exists: team14-airbnb-instant-booking-Triggered


## 5. Build the SageMaker Pipeline

In [5]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

from sagemaker.workflow.parameters import (
    ParameterFloat,
    ParameterInteger,
    ParameterString,
)

pipeline_session = PipelineSession()

# -------------------------------------------------------------------------
# Pipeline parameters — can be overridden at execution time
# -------------------------------------------------------------------------
p_input_data = ParameterString(name="InputDataUrl", default_value=SYNTHETIC_DATA_URI)
p_n_est = ParameterInteger(name="NEstimators", default_value=150)
p_depth = ParameterInteger(name="MaxDepth", default_value=20)
p_features = ParameterString(name="MaxFeatures", default_value="sqrt")
p_min_leaf = ParameterInteger(name="MinSamplesLeaf", default_value=2)
p_min_split = ParameterInteger(name="MinSamplesSplit", default_value=5)
p_n_jobs = ParameterInteger(name="NJobs", default_value=2)
p_rand_state = ParameterInteger(name="RandomState", default_value=42)

# Evaluation metric threshold (optional)
p_gate = ParameterFloat(name="QualityGateAUC", default_value=QUALITY_GATE_AUC)

print("Pipeline parameters defined.")

Pipeline parameters defined.


In [6]:
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep

# ==============================================================================
# Step 1: ProcessingStep (Data Cleaning & Feature Engineering)
# ==============================================================================

# 1. Initialize the Scikit-learn Processor
processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-process",
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

# 2. Define the ProcessingStep
step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=p_input_data, # <--- CHANGED FROM RAW_DATA_URI
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="processed",
            source="/opt/ml/processing/output",
            destination=f"{PIPELINE_ROOT}/processed",
        )
    ],
    code=f"{LOCAL_PIPELINE_SRC}/preprocess.py",
    job_arguments=[
        "--test-size", "0.2",
        "--random-state", "42",
    ],
)

print("Step 1 (ProcessingStep) defined.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [7]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

# ==============================================================================
# Step 2: TrainingStep (Tuned Random Forest Model)
# ==============================================================================
metric_definitions = [
    {
        "Name": "test_accuracy",
        "Regex": r"test_accuracy:\s*([0-9.]+)"
    },
    {
        "Name": "test_precision",
        "Regex": r"test_precision:\s*([0-9.]+)"
    },
    {
        "Name": "test_recall",
        "Regex": r"test_recall:\s*([0-9.]+)"
    },
    {
        "Name": "test_f1",
        "Regex": r"test_f1:\s*([0-9.]+)"
    },
    {
        "Name": "test_auc_roc",
        "Regex": r"test_auc_roc:\s*([0-9.]+)"
    }
]
# Estimator definition using the Scikit-learn container
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "max-features": p_features,
        "min-samples-leaf": p_min_leaf,
        "min-samples-split": p_min_split,
        "n-jobs": p_n_jobs,
        "random-state": p_rand_state,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    metric_definitions=metric_definitions,
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

# Reference the preprocessed dataset S3 URI from Step 1 (ProcessingStep)
processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

# Define the SageMaker Pipeline TrainingStep
step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
        "test": TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
    },
)

print("Step 2 (TrainingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [8]:
# Step 3: ModelStep — register in SageMaker Model Registry
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [9]:
# Step 4: ConditionStep — gate on SageMaker-captured test AUC
#
# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")

Step 4 (ConditionStep) defined.


In [10]:
# ==============================================================================
# Assemble and Upsert the SageMaker Pipeline
# ==============================================================================

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        p_input_data,
        p_n_est,
        p_depth,
        p_features,
        p_min_leaf,
        p_min_split,
        p_n_jobs,
        p_rand_state,
        p_gate,
    ],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)

print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print("View in SageMaker Studio: left sidebar -> Pipelines")

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team14-airbnb-instant-booking-trigger" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 7. Create a Synthetic CSV File

In [11]:
%%writefile SyntheticListing.csv
listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",,,,f,1,t,f,Buttes-Montmartre,,Paris,48.88668,2.33343,Entire apartment,Entire place,2,1,"[""Heating"", ""Kitchen"", ""Washer"", ""Wifi"", ""Long term stays allowed""]",53,2,1125,100,10,10,10,10,10,10,f
3705183,"Charming Studio near Eiffel Tower",10328771,2013-11-29,"Paris, Ile-de-France, France",within an hour,1.0,0.95,t,2,t,t,Passy,,Paris,48.85884,2.29435,Entire apartment,Entire place,2,1,"[""Shampoo"", ""Heating"", ""Kitchen"", ""Wifi"", ""Self check-in"", ""Keypad""]",120,2,1125,98,10,10,10,10,10,10,t
5128910,"Spacious Loft in Brooklyn with Skyline Views",22910481,2014-06-15,"New York, New York, United States",within a few hours,0.9,0.88,f,1,t,t,Williamsburg,,New York,40.71435,-73.95874,Entire loft,Entire place,4,2,"[""Air conditioning"", ""Kitchen"", ""Wifi"", ""Dedicated workspace"", ""Smart lock""]",185,3,30,95,9,10,10,9,9,9,t
6291024,"Cozy Manhattan Private Room near Central Park",18201945,2015-01-20,"New York, New York, United States",within a day,0.8,0.75,f,3,t,f,Upper West Side,,New York,40.78701,-73.97537,Private room in apartment,Private room,1,1,"[""Wifi"", ""Heating"", ""Air conditioning"", ""Lock on bedroom door""]",75,1,14,91,9,9,9,9,10,9,f
7401928,"Modern Condo with Ocean View in Bondi",39102914,2016-08-11,"Sydney, New South Wales, Australia",within an hour,1.0,1.0,t,4,t,t,Bondi Beach,,Sydney,-33.89147,151.27668,Entire condominium,Entire place,3,1,"[""Pool"", ""Kitchen"", ""Wifi"", ""Washer"", ""Dryer"", ""Self check-in""]",210,2,60,99,10,10,10,10,10,10,t
8192019,"Historic Roman Apartment near Colosseum",11029482,2012-04-05,"Rome, Lazio, Italy",,,,f,1,t,t,Monti,,Rome,41.89021,12.49223,Entire apartment,Entire place,4,2,"[""Heating"", ""Kitchen"", ""Wifi"", ""Air conditioning"", ""Washer""]",110,2,1125,94,9,9,10,10,10,9,f
9201948,"Luxury High-Rise Suite near Sukhumvit BTS",50192841,2017-09-22,"Bangkok, Bangkok, Thailand",within an hour,1.0,0.98,t,8,t,t,Khlong Toei,,Bangkok,13.73671,100.56108,Entire condominium,Entire place,2,1,"[""Gym"", ""Pool"", ""Wifi"", ""Air conditioning"", ""Building staff"", ""Keypad""]",1250,1,1125,97,10,10,10,10,10,10,t
10394819,"Sunny Copacabana Beachfront Apartment",29104820,2015-12-01,"Rio de Janeiro, Rio de Janeiro, Brazil",within a few hours,0.85,0.8,f,2,t,t,Copacabana,,Rio de Janeiro,-22.97112,-43.18224,Entire apartment,Entire place,5,2,"[""Wifi"", ""Kitchen"", ""Air conditioning"", ""Elevator"", ""Washer""]",320,3,90,92,9,9,9,9,10,9,f
11482019,"Traditional Flat in Historic Sultanahmet",44019283,2018-03-14,"Istanbul, Istanbul, Turkey",within an hour,0.95,0.92,f,5,t,f,Fatih,,Istanbul,41.00542,28.97681,Entire rental unit,Entire place,3,1,"[""Heating"", ""Wifi"", ""Air conditioning"", ""TV"", ""Hot water kettle""]",650,1,365,96,10,9,10,10,10,10,t
12591024,"Designer Loft in Roma Norte with Balcony",61029481,2019-05-19,"Mexico City, Distrito Federal, Mexico",within an hour,1.0,0.96,t,3,t,t,Cuauhtemoc,,Mexico City,19.41829,-99.16241,Entire loft,Entire place,2,1,"[""Wifi"", ""Dedicated workspace"", ""Kitchen"", ""Smart lock"", ""Balcony""]",1400,2,30,99,10,10,10,10,10,10,t
13692019,"Secluded Camps Bay Villa with Mountain Views",72019482,2016-10-30,"Cape Town, Western Cape, South Africa",within a few hours,0.9,0.85,t,2,t,t,Camps Bay,,Cape Town,-33.95124,18.37891,Entire villa,Entire place,6,3,"[""Pool"", ""Kitchen"", ""Wifi"", ""Free parking on premises"", ""Hot tub""]",3500,4,60,98,10,10,10,10,10,9,f
14782910,"Central Studio in Hong Kong Island",83019284,2020-01-15,"Hong Kong, Hong Kong",within an hour,1.0,1.0,f,1,t,t,Central & Western,,Hong Kong,22.28193,114.15821,Entire rental unit,Entire place,2,1,"[""Wifi"", ""Air conditioning"", ""Elevator"", ""Self check-in"", ""Keypad""]",580,1,180,95,9,10,10,10,10,9,t

Overwriting SyntheticListing.csv


In [12]:
# import time

# # Create your dynamic variable
# timestamp = int(time.time())
# filename = f"SyntheticListing_{timestamp}.csv"

# # The data you want to write
# csv_data = """listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
# 281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",,,,f,1,t,f,Buttes-Montmartre,,Paris,48.88668,2.33343,Entire apartment,Entire place,2,1,"[""Heating"", ""Kitchen"", ""Washer"", ""Wifi"", ""Long term stays allowed""]",53,2,1125,100,10,10,10,10,10,10,f
# 3705183,"Charming Studio near Eiffel Tower",10328771,2013-11-29,"Paris, Ile-de-France, France",within an hour,1.0,0.95,t,2,t,t,Passy,,Paris,48.85884,2.29435,Entire apartment,Entire place,2,1,"[""Shampoo"", ""Heating"", ""Kitchen"", ""Wifi"", ""Self check-in"", ""Keypad""]",120,2,1125,98,10,10,10,10,10,10,t
# """

# # Write the data to the dynamic filename
# with open(filename, "w") as f:
#     f.write(csv_data)

# print(f"File successfully written to: {filename}")

## 7.1 Verify the Local CSV File

In [13]:
import os
import pandas as pd

print("File exists:", os.path.exists(LOCAL_LISTING_FILE))
print("File size:", os.path.getsize(LOCAL_LISTING_FILE), "bytes")

df = pd.read_csv(LOCAL_LISTING_FILE, header=None)

print("Shape:", df.shape)
display(df.head())
print (df.shape)
if df.shape[1] != 33:
    raise ValueError("Expected 3 columns for the SyntheticListing CSV format.")

if df.iloc[:, -1].nunique() < 2:
    raise ValueError("Expected both target classes 0 and 1.")

File exists: True
File size: 4336 bytes
Shape: (13, 33)


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
0,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,...,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
1,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1,...,2,1125,100,10,10,10,10,10,10,f
2,3705183,Charming Studio near Eiffel Tower,10328771,2013-11-29,"Paris, Ile-de-France, France",within an hour,1.0,0.95,t,2,...,2,1125,98,10,10,10,10,10,10,t
3,5128910,Spacious Loft in Brooklyn with Skyline Views,22910481,2014-06-15,"New York, New York, United States",within a few hours,0.9,0.88,f,1,...,3,30,95,9,10,10,9,9,9,t
4,6291024,Cozy Manhattan Private Room near Central Park,18201945,2015-01-20,"New York, New York, United States",within a day,0.8,0.75,f,3,...,1,14,91,9,9,9,9,10,9,f


(13, 33)


## 8. Manual Pipeline Test

In [14]:
import time
import os
import shutil

# 1. Original local file
local_file = "SyntheticListing.csv"

# 2. Check if it exists
if not os.path.exists(local_file):
    raise FileNotFoundError(
        f"{local_file} not found. Run the CSV creation cell first."
    )

# 3. Create timestamp
timestamp = int(time.time())

# 4. Create timestamped local copy in CURRENT directory
local_copy = f"SyntheticListing_{timestamp}.csv"
shutil.copy2(local_file, local_copy)

# 5. Create timestamped S3 key
manual_s3_key = f"{MANUAL_TEST_PREFIX}/SyntheticListing_{timestamp}.csv"

# 6. Upload timestamped copy to S3
s3.upload_file(
    local_copy,
    BUCKET,
    manual_s3_key
)

# 7. S3 URI
manual_s3_uri = f"s3://{BUCKET}/{manual_s3_key}"

# 8. Print exact locations
print("========================================")
print("LOCAL COPY:")
print(os.path.abspath(local_copy))

print("\nS3 COPY:")
print(manual_s3_uri)
print("========================================")

LOCAL COPY:
/home/sagemaker-user/JG-Assignment/SyntheticListing_1787663127.csv

S3 COPY:
s3://nyp-26s1-iti113/iti113/team14/manual-input/SyntheticListing_1787663127.csv


In [15]:
import time
from sagemaker.workflow.pipeline import Pipeline # v2 import path

# Define the execution name
execution_name = f"manual-triggered-test-{int(time.time())}"

# 1. Attach to your already-deployed pipeline by its name
pipeline = Pipeline(
    name=PIPELINE_NAME,
    sagemaker_session=pipeline_session
)

# 2. Start the execution with a Python dictionary of parameters
execution = pipeline.start(
    execution_display_name=execution_name,
    parameters={
        "InputDataUrl": manual_s3_uri,
        "NEstimators": 150,
        "MaxDepth": 20,
        "MaxFeatures": "sqrt",
        "MinSamplesLeaf": 2,
        "MinSamplesSplit": 5,
        "NJobs": 2,
        "RandomState": 42,
        "QualityGateAUC": 0.75,
    }
)

# 3. Retrieve the ARN
TRIGGERED_EXECUTION_ARN = execution.arn
print("Manual pipeline execution started:")
print(TRIGGERED_EXECUTION_ARN)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Manual pipeline execution started:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team14-airbnb-instant-booking-trigger/execution/eyr4rkpptom3


## 8.1 Poll a Pipeline Execution

In [16]:
import time

def poll_pipeline_execution(execution_arn, sleep_seconds=30):
    """
    Poll a SageMaker Pipeline execution until it reaches a terminal status.
    """
    terminal_statuses = ["Succeeded", "Failed", "Stopped"]

    while True:
        desc = sm.describe_pipeline_execution(
            PipelineExecutionArn=execution_arn
        )

        status = desc["PipelineExecutionStatus"]

        print("=" * 80)
        print("Pipeline status:", status)
        print("Execution display name:", desc.get("PipelineExecutionDisplayName"))
        print("Start time:", desc.get("CreationTime") or desc.get("StartTime"))

        steps_response = sm.list_pipeline_execution_steps(
            PipelineExecutionArn=execution_arn
        )

        steps = steps_response.get("PipelineExecutionSteps", [])

        if steps:
            print("\nSteps:")
            for step in reversed(steps):
                step_name = step.get("StepName")
                step_status = step.get("StepStatus")
                failure_reason = step.get("FailureReason", "")

                print(f"- {step_name}: {step_status}")

                if failure_reason:
                    print(f"  Failure reason: {failure_reason}")
        else:
            print("\nNo steps listed yet.")

        if status in terminal_statuses:
            print()
            print("Final status:", status)
            return status

        print()
        print(f"Still running. Checking again in {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)

In [17]:
manual_status = poll_pipeline_execution(
    TRIGGERED_EXECUTION_ARN,
    sleep_seconds=30
)

Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00



No steps listed yet.

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


...

Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Succeeded
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Succeeded
- RegisterModel-RegisterModel: Succeeded

Final status: Succeeded


## 9. S3 Trigger Test

In [18]:
import time
import os

if not os.path.exists(LOCAL_LISTING_FILE):
    raise FileNotFoundError(
        f"{LOCAL_LISTING_FILE} not found. Run the CSV creation cell first."
    )

timestamp = int(time.time())

trigger_s3_key = (
    f"{TRIGGER_PREFIX}/"
    f"{LOCAL_LISTING_FILE}_{timestamp}.csv"
)

s3.upload_file(
    LOCAL_LISTING_FILE,
    BUCKET,
    trigger_s3_key
)

trigger_s3_uri = f"s3://{BUCKET}/{trigger_s3_key}"

print("Uploaded trigger test file:")
print(trigger_s3_uri)
print()
print("This should trigger:")
print("S3 → EventBridge → Lambda → SageMaker Pipeline")

Uploaded trigger test file:
s3://nyp-26s1-iti113/iti113/team14/trigger/input/SyntheticListing.csv_1787663671.csv

This should trigger:
S3 → EventBridge → Lambda → SageMaker Pipeline


In [19]:
import time

# Give EventBridge/Lambda a short time to start the pipeline execution.
time.sleep(10)

response = sm.list_pipeline_executions(
    PipelineName=PIPELINE_NAME,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)

if not response["PipelineExecutionSummaries"]:
    raise RuntimeError(f"No pipeline executions found for {PIPELINE_NAME}")

latest_execution_arn = response["PipelineExecutionSummaries"][0]["PipelineExecutionArn"]

print("Latest pipeline execution:")
print(latest_execution_arn)
print()

trigger_status = poll_pipeline_execution(
    latest_execution_arn,
    sleep_seconds=30
)

Latest pipeline execution:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team14-airbnb-instant-booking-trigger/execution/eyr4rkpptom3

Pipeline status: Succeeded
Execution display name: manual-triggered-test-1787663127
Start time: 2026-08-25 13:05:27.397000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- AUCQualityGate: Succeeded
- RegisterModel-RepackModel-0: Succeeded
- RegisterModel-RegisterModel: Succeeded

Final status: Succeeded


## 10. Result and Takeaway

If the final polling cell shows all steps succeeded, the event-driven retraining workflow is working.

Expected successful steps:

Pipeline Step	Expected Status
PreprocessData	Succeeded
TrainModel	Succeeded
EvaluateModel	Succeeded
CheckAUCQualityGate	Succeeded
